SPAM DETECTION: ML MODEL COMPARISON & TRAINING Notebook

This notebook compares 3 classification models:
1. Naive Bayes
2. Random Forest
3. Support Vector Machine (SVM)

Then selects the best model and trains it on your spam dataset.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from scipy import sparse
import matplotlib.pyplot as plt

In [ ]:
# Preprocess and clean dataset
def preprocess_and_clean(input_file):
    """Preprocess and clean spam data"""
    print("Loading data...")
    data = pd.read_csv(input_file)

    # Detect columns
    if 'v1' in data.columns and 'v2' in data.columns:
        text_col, label_col = 'v2', 'v1'
    else:
        text_col, label_col = data.columns[1], data.columns[0]

    print(f"Original shape: {data.shape}")

    # Clean text
    def clean_text(text):
        if pd.isna(text):
            return ""
        text = str(text).lower()
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    data['cleaned_text'] = data[text_col].apply(clean_text)
    data = data[data['cleaned_text'].str.strip() != '']

    # Convert labels
    data['label'] = data[label_col].map({'ham': 0, 'spam': 1})

    # Extract features
    data['text_length'] = data['cleaned_text'].str.len()
    data['word_count'] = data['cleaned_text'].apply(lambda x: len(str(x).split()))
    data['uppercase_ratio'] = data[text_col].apply(
        lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
    )

    # Remove duplicates
    initial = len(data)
    data = data.drop_duplicates(subset=['cleaned_text'])
    print(f"Removed {initial - len(data)} duplicates")

    # Handle outliers (simple: remove extreme values)
    for col in ['text_length', 'word_count']:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        data = data[(data[col] >= lower) & (data[col] <= upper)]

    print(f"Final shape: {data.shape}")

    return data

In [ ]:
# Vectorise and save dataset
def vectorise_and_save(data):
    """Create features and save them"""
    print("\nVectorising features...")

    # TF-IDF
    tfidf = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1,2))
    X_tfidf = tfidf.fit_transform(data['cleaned_text'])

    # Numeric features
    scaler = StandardScaler()
    X_numeric = scaler.fit_transform(data[['text_length', 'word_count', 'uppercase_ratio']])

    # Combine
    X = sparse.hstack([X_tfidf, X_numeric])
    y = data['label'].values

    print(f"Features shape: {X.shape}")

    # Save
    with open('features.pkl', 'wb') as f:
        pickle.dump(X, f)

    pd.DataFrame({'label': y}).to_csv('labels.csv', index=False)

    with open('tfidf_vectoriser.pkl', 'wb') as f:
        pickle.dump(tfidf, f)

    with open('scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)

    print("Saved: features.pkl, labels.csv, tfidf_vectoriser.pkl, scaler.pkl")

    return X, y

In [ ]:
# Load and train ML models
def load_features():
    """Load saved features"""
    print("Loading features...")

    with open('features.pkl', 'rb') as f:
        X = pickle.load(f)

    y = pd.read_csv('labels.csv')['label'].values

    print(f"Loaded: X={X.shape}, y={len(y)}")
    return X, y

def train_models(X, y):
    """Train and compare models"""
    print("\n" + "="*50)
    print("TRAINING MODELS")
    print("="*50)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"Train: {X_train.shape}, Test: {X_test.shape}")

    # Define models
    models = {
        'Naive Bayes': MultinomialNB(),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
    }

    results = {}

    # Train each model
    for name, model in models.items():
        print(f"\n=== {name} ===")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        results[name] = {
            'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1,
            'model': model, 'y_pred': y_pred, 'y_pred_proba': y_pred_proba
        }

        print(f"Accuracy : {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall   : {rec:.4f}")
        print(f"F1-score : {f1:.4f}")

    # Summary comparison table
    print("\n" + "="*70)
    print("MODEL COMPARISON SUMMARY")
    print("="*70)
    print(f"{'Model':<20} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
    print("-"*70)
    for name, res in results.items():
        print(f"{name:<20} {res['acc']:<12.4f} {res['prec']:<12.4f} {res['rec']:<12.4f} {res['f1']:<12.4f}")

    # Best model
    best_name = max(results, key=lambda x: results[x]['f1'])
    best_model = results[best_name]['model']
    best_res = results[best_name]

    print(f"BEST MODEL: {best_name} (F1-Score: {best_res['f1']:.4f})")

    # Plot ROC and Precision-Recall curves for best model
    plot_performance_curves(y_test, best_res['y_pred_proba'], best_name)

    # Final evaluation
    y_pred = best_model.predict(X_test)
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

    return best_model, results

def plot_performance_curves(y_test, y_pred_proba, model_name):
    """Plot ROC and Precision-Recall curves"""
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

    # Calculate metrics
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)

    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    ap = average_precision_score(y_test, y_pred_proba)

    # Create plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # ROC Curve
    ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC={roc_auc:.3f}')
    ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax1.set_xlim([0.0, 1.0])
    ax1.set_ylim([0.0, 1.05])
    ax1.set_xlabel('False Positive Rate', fontsize=12)
    ax1.set_ylabel('True Positive Rate', fontsize=12)
    ax1.set_title('ROC Curve (Test Set)', fontsize=14, fontweight='bold')
    ax1.legend(loc="lower right", fontsize=11)
    ax1.grid(alpha=0.3)

    # Precision-Recall Curve
    ax2.plot(recall, precision, color='darkorange', lw=2, label=f'AP={ap:.3f}')
    ax2.set_xlim([0.0, 1.0])
    ax2.set_ylim([0.0, 1.05])
    ax2.set_xlabel('Recall', fontsize=12)
    ax2.set_ylabel('Precision', fontsize=12)
    ax2.set_title('Precision-Recall Curve (Test Set)', fontsize=14, fontweight='bold')
    ax2.legend(loc="upper right", fontsize=11)
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('model_performance_curves.png', dpi=150, bbox_inches='tight')
    print("\nPerformance curves saved as 'model_performance_curves.png'")
    plt.show()

    print(f"\nROC-AUC Score: {roc_auc:.4f}")
    print(f"Average Precision Score: {ap:.4f}")

In [ ]:
if __name__ == "__main__":
    import sys

    if len(sys.argv) > 1:
        step = sys.argv[1]

        if step == "preprocess":
            # Step 1: Preprocess and clean
            data = preprocess_and_clean('spam_emails.csv')
            X, y = vectorise_and_save(data)

        elif step == "train":
            # Step 2: Load and train
            X, y = load_features()
            best_model, results = train_models(X, y)

    else:
        # Run everything
        print("STEP 1: PREPROCESSING & CLEANING")
        print("="*50)
        data = preprocess_and_clean('spam_emails.csv')

        print("\nSTEP 2: VECTORISING & SAVING")
        print("="*50)
        X, y = vectorise_and_save(data)

        print("\nSTEP 3: TRAINING MODELS")
        print("="*50)
        best_model, results = train_models(X, y)

        print("\nPIPELINE COMPLETE!")